In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib.colors import LinearSegmentedColormap
from collections import defaultdict
import matplotlib
# ----------------------------
# Load data
# ----------------------------
names_file = "../../../data/graphing/iMM/IMMDataNames.xlsx"
classification_file = "../../../data/graphing/iMM/IMMDataNames_with_classification_and_stats.xlsx"
names_xls = pd.ExcelFile(names_file)
map_df = pd.read_excel(classification_file, sheet_name='MetaboliteClassification')
map_df = map_df[~map_df['Name'].str.strip().str.lower().eq("h2o")]  # remove H2O

# ----------------------------
# Phylum dictionary
# ----------------------------

phyla_dict = {
    "cohm": "Firmicutes",
    "cuvk": "Actinobacteria",
    "cxy": "Actinobacteria",
    "mip": "Actinobacteria",
    "rfa": "Actinobacteria",
    "cpi": "Bacteroidetes",
    "flk": "Bacteroidetes",
    "mgk": "Bacteroidetes",
    "bbet": "Alphaproteobacteria",
    "buol": "Betaproteobacteria",
    "cauj": "Alphaproteobacteria",
    "dri": "Alphaproteobacteria",
    "lgu": "Gammaproteobacteria",
    "lur": "Gammaproteobacteria",
    "mlot": "Alphaproteobacteria",
    "pmor": "Gammaproteobacteria",
    "pscf": "Gammaproteobacteria",
    "pzu": "Alphaproteobacteria",
    "racd": "Gammaproteobacteria",
    "tro": "Alphaproteobacteria",
    "vpe": "Betaproteobacteria",
}
phyla_order = [
    "Alphaproteobacteria",
    "Gammaproteobacteria",
    "Actinobacteria",
    "Bacteroidetes",
    "Betaproteobacteria",
    "Firmicutes",
    "Unknown"
]

# ----------------------------
# Compute metabolite frequencies
# ----------------------------

def compute_presence_matrix(names_xls, mode='min'):
    presence_counts = defaultdict(lambda: defaultdict(int))
    total_counts = {}

    for sheet in names_xls.sheet_names:
        model = sheet.replace("_summary", "")
        df = pd.read_excel(names_xls, sheet_name=sheet)
        if 'Row' in df.columns:
            df = df.drop(columns=['Row'])

        if mode == 'min':
            mask = df.applymap(lambda x: isinstance(x, str) and x.strip().lower() == 'none').any(axis=1)
            if mask.sum() == 0:
                mask[:] = True  # interpret as min if no 'none'
        else:
            mask = ~df.applymap(lambda x: isinstance(x, str) and x.strip().lower() == 'none').any(axis=1)

        selected_df = df[mask].dropna(how='all')
        total_counts[model] = len(selected_df)

        for _, row in selected_df.iterrows():
            for met in row.dropna():
                if isinstance(met, str) and met.strip().lower() != 'h2o':  # remove H2O
                    presence_counts[model][met.strip()] += 1

    all_models = sorted(total_counts.keys())
    all_mets = sorted(set(m for model_data in presence_counts.values() for m in model_data))
    matrix = pd.DataFrame(index=all_mets, columns=all_models)

    for model in all_models:
        total = total_counts[model]
        for met in all_mets:
            matrix.at[met, model] = presence_counts[model].get(met, 0) / total if total > 0 else 0

    matrix = matrix.astype(float)
    matrix = matrix.drop(index='none', errors='ignore')
    return matrix, total_counts

# ----------------------------
# Plot heatmap and barplot
# ----------------------------

def plot_results(presence_matrix, total_counts, map_df, bar_color="Blues", heatmap_color=("#ffffff", "#104547"), title="Min"):
    classes = (
        map_df[map_df["Name"].isin(presence_matrix.index)]
        .drop_duplicates("Name")
        .set_index("Name")
        .reindex(presence_matrix.index)["Class"]
        .fillna("Unknown")
    )

    # Sort rows by class
    matrix = presence_matrix.assign(Class=classes).sort_values("Class")
    class_labels = matrix.pop("Class")

    # Assign phylum to models
    model_phyla = pd.Series({m: phyla_dict.get(m, "Unknown") for m in matrix.columns})
    matrix.columns = pd.MultiIndex.from_arrays([model_phyla, matrix.columns])

    # Reorder models by phylum
    matrix = matrix.reindex(columns=pd.MultiIndex.from_tuples([
        (phy, m)
        for phy in phyla_order
        for m in matrix.columns.get_level_values(1)
        if (phy, m) in matrix.columns
    ]))

    # Barplot
    ordered_models = matrix.columns.get_level_values(1)
    counts = [total_counts[m] for m in ordered_models]
    norm = plt.Normalize(min(counts), max(counts))
    cmap_bar = plt.get_cmap(bar_color)
    bar_cols = [cmap_bar(norm(v)) for v in counts]

    plt.figure(figsize=(20, 4))
    plt.bar(ordered_models, counts, color=bar_cols)
    plt.ylabel(f"# {title} alternatives", fontsize=14)
    plt.xticks(rotation=45, ha="right", fontsize=12)
    plt.yticks(fontsize=12)
    plt.title(f"{title} alternatives per model (Grouped by Phylum)", fontsize=16)
    plt.tight_layout()
    plt.show()

    # Heatmap
    class_palette = dict(zip(class_labels.unique(), sns.color_palette("tab10", len(class_labels.unique()))))
    row_colors = class_labels.map(class_palette)

    plt.figure(figsize=(20, 22))
    sns.heatmap(
        matrix,
        cmap=LinearSegmentedColormap.from_list(f"cm_{title}", heatmap_color),
        linewidths=0.5,
        linecolor="gray",
        cbar_kws={"shrink": 0.5, "label": "Relative frequency"},
    )
    plt.xlabel("")
    plt.ylabel("")
    plt.xticks(
        ticks=np.arange(len(matrix.columns)) + 0.5,
        labels=matrix.columns.get_level_values(1),
        rotation=45,
        ha="right",
        fontsize=12,
    )
    plt.yticks(fontsize=12)

    handles = [
        plt.Line2D([0], [0], marker="s", color=c, label=cls, linestyle="", markersize=10)
        for cls, c in class_palette.items()
    ]
    plt.legend(
        handles=handles,
        title="Class",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=12,
        title_fontsize=14,
    )
    plt.title(f"{title}: metabolite usage across models (by Phylum)", fontsize=16)
    plt.tight_layout()
    plt.savefig(f"{title}_heatmap.pdf", dpi=300, bbox_inches="tight")  # Save as PDF
    plt.show()

# ----------------------------
# Run and plot
# ----------------------------

min_matrix, min_counts = compute_presence_matrix(names_xls, mode='min')
plot_results(min_matrix, min_counts, map_df, bar_color="Blues", heatmap_color=("#ffffff", "#104547"), title="Min")
minp1_matrix, minp1_counts = compute_presence_matrix(names_xls, mode='minp1')
plot_results(minp1_matrix, minp1_counts, map_df, bar_color="Oranges", heatmap_color=("#ffffff", "#c63c00"), title="Min+1")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import seaborn as sns
import matplotlib
matplotlib.use('cairo')  # Use a non-interactive backend
# Data
alternatives = [20, 8, 16, 6, 2, 8, 20, 8, 9, 2, 4, 20, 8, 24, 40, 12, 2, 16, 4, 120, 2]
immsize      = [10,10,10,10, 8, 7, 10, 9, 9, 8, 8,  9, 9, 16, 11, 10,10, 9, 8,  10, 10]
data = np.array([alternatives, immsize], dtype=float)
N = data.shape[1]

# Masks (True = hide)
mask_alt = np.ones_like(data, dtype=bool); mask_alt[0, :] = False  # show only row 0
mask_imm = np.ones_like(data, dtype=bool); mask_imm[1, :] = False  # show only row 1

fig, ax = plt.subplots(figsize=(12, 3))

# Plot row 0 with Blues, row 1 with Purples
sns.heatmap(
    data, mask=mask_alt, cmap="Blues",
    vmin=min(alternatives), vmax=max(alternatives),
    cbar=False, ax=ax, square=False
)
sns.heatmap(
    data, mask=mask_imm, cmap="Purples",
    vmin=min(immsize), vmax=max(immsize),
    cbar=False, ax=ax, square=False
)

# Ticks/labels
ax.set_yticks([0.5, 1.5])
ax.set_yticklabels(["Alternatives", "IMMSize"])
ax.set_xticks(np.arange(N) + 0.5)
ax.set_xticklabels([str(i) for i in range(1, N + 1)], rotation=0)
ax.set_xlabel("Column index")
ax.set_title("Seaborn heatmap with separate colormaps")

# Two separate colorbars
divider = make_axes_locatable(ax)
cax_top = divider.append_axes("top", size="6%", pad=0.35)
cax_bot = divider.append_axes("bottom", size="6%", pad=0.45)

sm_alt = ScalarMappable(Normalize(min(alternatives), max(alternatives)), cmap="Blues")
sm_imm = ScalarMappable(Normalize(min(immsize), max(immsize)), cmap="Purples")
sm_alt.set_array([]); sm_imm.set_array([])

cbar_alt = fig.colorbar(sm_alt, cax=cax_top, orientation="horizontal")
cbar_alt.set_label("Alternatives")
cax_top.xaxis.set_ticks_position("top")
cax_top.xaxis.set_label_position("top")

cbar_imm = fig.colorbar(sm_imm, cax=cax_bot, orientation="horizontal")
cbar_imm.set_label("IMMSize")

plt.tight_layout()
plt.savefig("IMMStats.pdf", dpi=300, bbox_inches="tight")  # Save as PDF
plt.show()
